# Simplified path — Colab GPU run

Runs `src/simplified.py` end-to-end on a Colab GPU. Single model: Gemma 2-2B. Same code path that runs on a local Mac CPU; the GPU just makes it faster.

**Before running:** Runtime → Change runtime type → GPU (T4 is more than enough for Gemma 2-2B in fp16).

Outputs are written to `colab_outputs.md` in the working directory; download from the Files panel after the run.

## 1. Clone repo & install dependencies

In [ ]:
# Cloning from b-coman fork simplified-cleanup branch.
# Once the PR is merged into dariacoman/main, switch this clone URL.
!git clone --branch simplified-cleanup https://github.com/b-coman/compliance-gap-analysis.git
%cd compliance-gap-analysis

In [ ]:
# Install only what Colab does not preinstall.
# Colab already ships torch, numpy, etc.
# numpy<2.2 keeps ABI consistent with Colab's preinstalled scipy/sklearn.
!pip install -q sentence-transformers transformers accelerate diskcache "numpy<2.2"

## 2. HuggingFace authentication

The model used in this project is `google/gemma-2-2b-it` — a gated model on HuggingFace. Before running the next cells:

1. Accept the licence on https://huggingface.co/google/gemma-2-2b-it (one-click).
2. Generate a read-token at https://huggingface.co/settings/tokens.
3. Add it as a Colab secret (left sidebar → key icon → name `HF_TOKEN`, value the token).

The next cell will load it into the environment so transformers can fetch the model.

In [ ]:
import os

from google.colab import userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded.')
except Exception:
    print('WARNING: HF_TOKEN not found. Add it as a Colab secret to load Gemma 2.')

os.environ['PYTHONPATH'] = '/content/compliance-gap-analysis'

## 3. Verify GPU & import the simplified path

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('Memory:', torch.cuda.get_device_properties(0).total_memory / 1e9, 'GB')

In [ ]:
from src.simplified import analyse, LLM_MODEL_ID
print('Model in use:', LLM_MODEL_ID)

## 4. Run the 5 standard test queries

Verbatim text from `docs/test-queries.md`. These are the same queries run in the local baseline (`docs/test-passes/v3-qwen-1.5b-local-baseline.md`) — outputs from this notebook are directly comparable.

In [ ]:
QUERIES = {
    'Q1 multi-facet': (
        "TalentLens compliance under EU AI Act Annex III \u00a74 \u2014 am I covered on "
        "Article 13 deployer instructions, Article 14 human oversight, Article 26 logs "
        "and worker information, and the related Article 22 GDPR automated-decisions "
        "duties? Where are the gaps?"
    ),
    'Q2 red-teaming': (
        "Does our policy address the red-teaming requirements before deploying a high-"
        "risk AI system to production?"
    ),
    'Q3 Article 22 sub-clauses': (
        "How do we meet GDPR Article 22 requirements on solely automated decisions "
        "affecting candidates \u2014 explicit consent, right to obtain human intervention, "
        "right to contest the decision, and right to express their point of view?"
    ),
    'Q4 transparency': (
        "Are we doing enough on transparency for candidates assessed by TalentLens?"
    ),
    'Q5 FRIA': (
        "Have we performed a Fundamental Rights Impact Assessment under EU AI Act "
        "Article 27 for TalentLens as a deployer of an Annex III high-risk system?"
    ),
}

In [ ]:
import time

results = {}
for label, query in QUERIES.items():
    print(f'\n{"="*70}\n{label}\n{"="*70}')
    print(f'Query: {query}\n')
    t0 = time.time()
    output = analyse(query)
    elapsed = time.time() - t0
    results[label] = {'query': query, 'output': output, 'elapsed_s': elapsed}
    print(output)
    print(f'\n[{elapsed:.1f}s]')

## 5. Save outputs to a markdown file

Download `colab_outputs.md` from the Files panel afterwards. Drop it into `docs/test-passes/` in the repo with a descriptive filename, e.g. `v4-qwen-7b-colab.md`.

In [ ]:
from datetime import datetime, timezone

lines = [
    f'# Test pass \u2014 simplified path on Colab GPU',
    '',
    f'> Run date: {datetime.now(timezone.utc).strftime("%Y-%m-%d")}',
    f'> Model: `{LLM_MODEL_ID}`',
    f'> Hardware: Colab GPU ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})',
    f'> Prompt: V4 (current default in `src/simplified.py`)',
    f'> Embeddings: BAAI/bge-large-en-v1.5',
    '',
    '---',
    '',
]
for label, r in results.items():
    lines += [
        f'## {label}',
        '',
        f'**Query:** {r["query"]}',
        '',
        f'**Generation latency:** {r["elapsed_s"]:.1f}s',
        '',
        '**Output:**',
        '',
        '```',
        r['output'].rstrip(),
        '```',
        '',
        '---',
        '',
    ]

with open('colab_outputs.md', 'w') as f:
    f.write('\n'.join(lines))

print('Wrote colab_outputs.md')